# Exploratory Data Analysis (EDA) Project

## Objective
Analyze a sales dataset to uncover patterns, trends, relationships, and key influencing factors using statistical summaries and visualizations.

This project demonstrates:
- Data inspection and quality assessment
- Descriptive statistics
- Missing-value and duplicate analysis
- Univariate, bivariate, and time-series analysis
- Correlation analysis
- Business insights and structured conclusions

**Workflow:** Raw Data → Data Quality → Statistical Summary → EDA → Visualizations → Insights


## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## 2. Load Dataset

In [ ]:
df = pd.read_csv("sales_eda_dataset.csv", parse_dates=["Order_Date"])

print("Dataset shape:", df.shape)
display(df.head())


## 3. Initial Data Inspection

In [ ]:
df.info()


In [ ]:
display(df.describe(include="all").T)


In [ ]:
print("Duplicate rows:", df.duplicated().sum())
display(df.isna().sum().to_frame("Missing Values"))


## 4. Data Quality Visualization

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]

plt.figure(figsize=(9,5))
sns.barplot(x=missing.values, y=missing.index)
plt.title("Missing Values by Column")
plt.xlabel("Number of Missing Values")
plt.ylabel("Column")
plt.show()


## 5. Data Preparation

In [ ]:
analysis_df = df.copy()

# Fill numerical missing values with median
for column in analysis_df.select_dtypes(include=np.number).columns:
    analysis_df[column] = analysis_df[column].fillna(
        analysis_df[column].median()
    )

# Fill categorical missing values with mode
for column in analysis_df.select_dtypes(exclude=np.number).columns:
    if column != "Order_Date":
        mode = analysis_df[column].mode()
        if not mode.empty:
            analysis_df[column] = analysis_df[column].fillna(mode.iloc[0])

analysis_df = analysis_df.drop_duplicates().reset_index(drop=True)

print("Original shape:", df.shape)
print("Analysis shape:", analysis_df.shape)
print("Remaining missing values:", analysis_df.isna().sum().sum())


## 6. Univariate Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,5))

sns.histplot(analysis_df["Revenue"], kde=True, ax=axes[0])
axes[0].set_title("Revenue Distribution")
axes[0].set_xlabel("Revenue")

sns.histplot(analysis_df["Customer_Age"], kde=True, ax=axes[1])
axes[1].set_title("Customer Age Distribution")
axes[1].set_xlabel("Customer Age")

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(9,5))
sns.countplot(data=analysis_df, x="Product_Category",
              order=analysis_df["Product_Category"].value_counts().index)
plt.title("Orders by Product Category")
plt.xlabel("Product Category")
plt.ylabel("Number of Orders")
plt.xticks(rotation=20)
plt.show()


## 7. Category and Regional Analysis

In [ ]:
category_revenue = (
    analysis_df.groupby("Product_Category")["Revenue"]
    .sum()
    .sort_values(ascending=False)
)

plt.figure(figsize=(9,5))
sns.barplot(x=category_revenue.values, y=category_revenue.index)
plt.title("Total Revenue by Product Category")
plt.xlabel("Revenue")
plt.ylabel("Product Category")
plt.show()


In [ ]:
region_revenue = (
    analysis_df.groupby("Region")["Revenue"]
    .sum()
    .sort_values(ascending=False)
)

plt.figure(figsize=(8,5))
sns.barplot(x=region_revenue.index, y=region_revenue.values)
plt.title("Total Revenue by Region")
plt.xlabel("Region")
plt.ylabel("Revenue")
plt.show()


In [ ]:
channel_revenue = (
    analysis_df.groupby("Sales_Channel")["Revenue"]
    .sum()
    .sort_values(ascending=False)
)

plt.figure(figsize=(8,5))
sns.barplot(x=channel_revenue.index, y=channel_revenue.values)
plt.title("Revenue by Sales Channel")
plt.xlabel("Sales Channel")
plt.ylabel("Revenue")
plt.show()


## 8. Time-Series Analysis

In [ ]:
monthly_revenue = (
    analysis_df.set_index("Order_Date")
    .resample("ME")["Revenue"]
    .sum()
)

plt.figure(figsize=(12,5))
monthly_revenue.plot(marker="o")
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.tight_layout()
plt.show()


In [ ]:
quarterly_revenue = (
    analysis_df.set_index("Order_Date")
    .resample("QE")["Revenue"]
    .sum()
)

display(quarterly_revenue.to_frame("Revenue"))


## 9. Relationship and Correlation Analysis

In [ ]:
numeric_columns = [
    "Customer_Age", "Units_Sold", "Unit_Price",
    "Discount", "Customer_Rating", "Revenue"
]

plt.figure(figsize=(9,7))
sns.heatmap(
    analysis_df[numeric_columns].corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)
plt.title("Correlation Matrix")
plt.show()


In [ ]:
plt.figure(figsize=(9,5))
sns.scatterplot(
    data=analysis_df,
    x="Units_Sold",
    y="Revenue",
    hue="Product_Category",
    alpha=.7
)
plt.title("Units Sold vs Revenue")
plt.xlabel("Units Sold")
plt.ylabel("Revenue")
plt.show()


In [ ]:
plt.figure(figsize=(9,5))
sns.boxplot(
    data=analysis_df,
    x="Product_Category",
    y="Revenue"
)
plt.title("Revenue Distribution by Product Category")
plt.xlabel("Product Category")
plt.ylabel("Revenue")
plt.xticks(rotation=20)
plt.show()


## 10. Statistical Business Summary

In [ ]:
summary = pd.DataFrame({
    "Metric": [
        "Total Revenue",
        "Average Order Revenue",
        "Median Order Revenue",
        "Total Units Sold",
        "Average Customer Rating",
        "Number of Orders",
        "Best Revenue Category",
        "Best Revenue Region",
        "Best Revenue Channel"
    ],
    "Value": [
        analysis_df["Revenue"].sum(),
        analysis_df["Revenue"].mean(),
        analysis_df["Revenue"].median(),
        analysis_df["Units_Sold"].sum(),
        analysis_df["Customer_Rating"].mean(),
        analysis_df["Order_ID"].nunique(),
        category_revenue.index[0],
        region_revenue.index[0],
        channel_revenue.index[0]
    ]
})

display(summary)


## 11. Key Insights

The analysis should be interpreted using the generated tables and visualizations.

Typical insights to discuss:

1. Which product category contributes the highest total revenue?
2. Which region and sales channel perform best?
3. Does revenue show an increasing, decreasing, or seasonal pattern over time?
4. How strongly are units sold and revenue related?
5. Which categories show the greatest revenue variability?
6. Are customer ratings concentrated around a particular range?
7. Are discounts associated with higher or lower revenue?

> Correlation indicates association, not causation. Business decisions should consider context, costs, and additional data.


## Conclusion

In [ ]:
print("EDA completed successfully.")
print("The dataset was inspected, cleaned, summarized, visualized, and interpreted.")
